# 01 — Exploratory Data Analysis

This notebook is the single source of truth for understanding the raw Amazon Reviews data before preprocessing or modelling. It inspects schema, class imbalance, missing values, duplicates, review length, text quality, and product metadata. No processed dataset is created here.

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
RAW_DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
TRAIN_PATH = RAW_DATA_DIR / 'train_data.csv'
TEST_PATH = RAW_DATA_DIR / 'test_data.csv'
PRODUCT_PATH = RAW_DATA_DIR / 'title_brand.csv'

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')
TRAIN_PATH

## Load raw datasets

In [ ]:
train_df = pd.read_csv(TRAIN_PATH, low_memory=False)
test_df = pd.read_csv(TEST_PATH, low_memory=False)
product_df = pd.read_csv(PRODUCT_PATH, low_memory=False)

pd.DataFrame({
    'dataset': ['train', 'test', 'product_metadata'],
    'rows': [len(train_df), len(test_df), len(product_df)],
    'columns': [train_df.shape[1], test_df.shape[1], product_df.shape[1]],
})

In [ ]:
display(train_df.head(3))
display(test_df.head(3))
display(product_df.head(3))
print('Train columns:', train_df.columns.tolist())
print('Test columns:', test_df.columns.tolist())
print('Product columns:', product_df.columns.tolist())

## Target distribution and imbalance

In [ ]:
target_counts = train_df['overall'].value_counts().sort_index()
target_summary = pd.DataFrame({
    'count': target_counts,
    'percentage': (target_counts / len(train_df) * 100).round(2),
})
display(target_summary)

ax = target_counts.plot(kind='bar', color=sns.color_palette('viridis', 5), figsize=(8, 4))
ax.set(title='Rating distribution in raw training data', xlabel='Rating', ylabel='Reviews')
ax.tick_params(axis='x', rotation=0)
plt.show()

## Missing values and duplicates

In [ ]:
missing_summary = pd.DataFrame({
    'missing_count': train_df.isna().sum(),
    'missing_percentage': (train_df.isna().mean() * 100).round(2),
}).sort_values('missing_count', ascending=False)
display(missing_summary)

print('Exact duplicate rows:', int(train_df.duplicated().sum()))
print('Duplicate reviewText values:', int(train_df['reviewText'].duplicated().sum()))
print('Missing reviewText values:', int(train_df['reviewText'].isna().sum()))

## Review-length analysis

In [ ]:
review_text = train_df['reviewText'].fillna('').astype(str)
review_length = review_text.str.len()
display(review_length.describe(percentiles=[0.50, 0.90, 0.95, 0.99, 0.995, 0.999]).to_frame('characters'))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(review_length, bins=100, ax=axes[0])
axes[0].set(title='Full review-length distribution', xlabel='Characters')
upper_99 = review_length.quantile(0.99)
sns.histplot(review_length[review_length <= upper_99], bins=100, ax=axes[1])
axes[1].set(title='Review lengths up to the 99th percentile', xlabel='Characters')
plt.tight_layout()
plt.show()

## Text-quality checks relevant to BERT-family models

In [ ]:
html_pattern = re.compile(r'<[^>]+>')
url_pattern = re.compile(r'https?://\S+|www\.\S+', flags=re.IGNORECASE)

text_quality = pd.Series({
    'empty_or_whitespace': int(review_text.str.strip().eq('').sum()),
    'contains_possible_html': int(review_text.str.contains(html_pattern, na=False).sum()),
    'contains_url': int(review_text.str.contains(url_pattern, na=False).sum()),
    'contains_line_break': int(review_text.str.contains(r'[\r\n]', regex=True, na=False).sum()),
}, name='row_count')
text_quality.to_frame()

## Product and brand coverage

In [ ]:
metadata_unique = product_df.drop_duplicates(subset='asin')
metadata_coverage = train_df['asin'].isin(metadata_unique['asin']).mean()
print(f'ASIN metadata coverage: {metadata_coverage:.2%}')

brand_summary = (
    train_df[['asin', 'overall']]
    .merge(metadata_unique[['asin', 'brand']], on='asin', how='left')
    .groupby('brand', dropna=False)
    .agg(review_count=('overall', 'size'), average_rating=('overall', 'mean'))
    .sort_values('review_count', ascending=False)
    .head(10)
)
brand_summary['average_rating'] = brand_summary['average_rating'].round(3)
brand_summary

## EDA conclusions

- The task is five-class rating prediction (`1..5`) from `reviewText`.
- The raw target is strongly imbalanced, especially toward rating 5.
- Exact duplicate rows must be removed before sampling and splitting.
- Reviews have a strongly right-skewed length distribution.
- Transformer preprocessing should remain light: remove HTML/URLs and normalize whitespace while preserving case, punctuation, sentence structure, stopwords, and negation.
- The untouched official test set is reserved only for final inference.